# CropMind AI — Baseline Training on Colab (free GPU route)

Runs the **real** Phase 3 training (`ml/configs/train_v1.yaml`, seed 42) on Colab's free T4 GPU
instead of a local CPU. Same code, same config, same deterministic splits — only the hardware differs.

**Cost:** £0. **Time:** ~5–10 min per epoch on T4 (12 epochs max; early stopping may finish sooner).

> **Honesty / provenance policy**
> - No number here is a validated metric until published in `reports/model_evaluation/` (Phase 4).
> - The dataset ZIP is uploaded by you to *your own* Drive (PlantVillage CC0 1.0 — keep the folder private, do not re-share).
> - Mendeley refuses automated download (HTTP 403) — this notebook never attempts it. It uses the same archive you already verified locally, optionally **pinned by sha256**.
> - Colab GPU runs are not bit-reproducible; the `metrics.json` written by *this* run is the canonical artifact behind this checkpoint.

**Prerequisites (one-time):**
1. Create a folder `cropmind` inside **My Drive**.
2. Upload `Plant_leaf_diseases_dataset_without_augmentation.zip` — your **existing** local download — into `My Drive/cropmind/` (do NOT re-download from Mendeley).
3. Optional but recommended: open your local `data\raw\plantvillage\PROVENANCE.json`, copy `acquisition.archive_sha256`, and paste it into the config cell as `EXPECTED_SHA256` (proves Colab trains on bytes identical to your verified import).
4. Runtime → Change runtime type → **T4 GPU**. Then run the cells top to bottom.

## Step 0 — GPU sanity

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU — set Runtime > Change runtime type > T4 GPU, then rerun all."
print("GPU:", torch.cuda.get_device_name(0))
print("torch:", torch.__version__)

## Step 1 — Mount Drive, set config, clone the pinned repo

In [ ]:
from pathlib import Path

DRIVE_ZIP = Path("/content/drive/MyDrive/cropmind/Plant_leaf_diseases_dataset_without_augmentation.zip")
EXPECTED_SHA256 = ""  # optional: paste acquisition.archive_sha256 from your local PROVENANCE.json
GIT_URL = "https://github.com/Vishwa-cloud25S/cropmind-ai.git"
COMMIT = "8778a8f5924c17a8a6d834a60b1467172e50d064"  # pinned main (2026-08-08); bump to a newer sha only if you pulled newer code

from google.colab import drive

drive.mount("/content/drive")
assert DRIVE_ZIP.exists(), (
    f"Archive not found at {DRIVE_ZIP}. Upload 'Plant_leaf_diseases_dataset_without_augmentation.zip' "
    "to My Drive/cropmind/ first (use your EXISTING local download — do not re-download)."
)

## Step 2 — Verify/pin the archive checksum

In [ ]:
import hashlib


def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


archive_sha = sha256_file(DRIVE_ZIP)
print("archive sha256:", archive_sha)
if EXPECTED_SHA256:
    assert archive_sha == EXPECTED_SHA256, "Drive archive differs from your verified local import — aborting."
    print("PINNED ✓ — identical to the sha256 recorded in your local PROVENANCE.json")
else:
    print("WARNING: EXPECTED_SHA256 unset — proceeding unpinned (allowed, but pinning is recommended).")

## Step 3 — Clone the repo at the pinned commit + install deps

In [ ]:
!git clone -q {GIT_URL} /content/cropmind-ai 2>/dev/null || git -C /content/cropmind-ai fetch -q origin
%cd /content/cropmind-ai
!git checkout -q {COMMIT}
!git log --oneline -1
!python -m pip install -q -r ml/requirements.txt  # Colab's GPU torch already satisfies torch>=2.2 — no CPU wheel is pulled

## Step 4 — Import → split → verify → stats (idempotent; no downloading)

Fails fast if any stage fails. Uses the without-augmentation tree only; rerun freely — extraction is reused.
Expect: `structure OK` … `verification OK` … ~27.3k images across 21 classes.

In [ ]:
!python -m ml.data.cli import --dataset plantvillage --archive "{DRIVE_ZIP}" --accept-license && \
  python -m ml.data.cli split --dataset plantvillage && \
  python -m ml.data.cli verify --dataset plantvillage && \
  python -m ml.data.cli stats --dataset plantvillage

## Step 5 — Train (leave this tab open)

~598 batches/epoch from 19,126 train images; one line per epoch below. Early stopping on val top-1
(patience 4) may end the run before epoch 12 — that is the scheduler doing its job, not a failure.
Artifacts are written only at completion, so do not interrupt mid-run.

In [ ]:
!python -m ml.training.train --config ml/configs/train_v1.yaml

## Step 6 — Results summary

In [ ]:
import glob
import json

metrics_path = sorted(glob.glob("runs/*/metrics.json"))[-1]
run_dir = Path(metrics_path).parent
m = json.loads(Path(metrics_path).read_text())
print("run_id:", m["run_id"])
print("best_val_top1:", round(m["best_val_top1"], 4))
print("held-out test top1:", round(m["test"]["top1"], 4), "| test loss:", round(m["test"]["loss"], 4))
print("dataset:", m["dataset"], "| splits sha256:", m["splits_content_sha256"][:16], "\u2026")
per = m["test"]["per_class"]
weakest = sorted(zip(m["classes"], per["f1"], strict=True), key=lambda kv: kv[1])[:5]
print("weakest classes by F1 (the full per-class table is published in the Phase 4 report):")
for cls, f1 in weakest:
    print(f"  {cls}: {f1:.3f}")

## Step 7 — Persist artifacts (Drive copy + auto-download)

In [ ]:
import shutil

drive_dest = Path(f"/content/drive/MyDrive/cropmind/runs/{run_dir.name}")
shutil.rmtree(drive_dest, ignore_errors=True)
shutil.copytree(run_dir, drive_dest)
print("copied to Drive:", drive_dest)

bundle = shutil.make_archive(f"/content/{run_dir.name}", "zip", run_dir)
from google.colab import files

files.download(bundle)  # ~10–15 MB: checkpoint.pt, checkpoint.sha256, metrics.json, config.yaml

## Step 8 — Checkpoint integrity sanity-check

In [ ]:
ckpt = torch.load(run_dir / "checkpoint.pt", map_location="cpu", weights_only=True)
meta = ckpt["meta"]
assert meta["checkpoint_sha256"] == (run_dir / "checkpoint.sha256").read_text().strip()
print(
    "checkpoint verified — version", meta["model_version"],
    "| classes", meta["num_classes"],
    "| val top1", round(meta["val_top1"], 4),
    "| test top1", round(meta["test_top1"], 4),
)

## Step 9 — Bring the results home (Windows)

1. `runs-<run_id>.zip` auto-downloaded in Step 7 (also in `My Drive/cropmind/runs/`).
2. Extract it into your local repo's `runs\` folder → you now have `runs\<run_id>\checkpoint.pt`
   + `metrics.json` locally (gitignored — artifacts never enter git).
3. The checkpoint's `splits_content_sha256` ties it to the exact verified split files — your local
   provenance chain stays intact.
4. Paste the contents of `runs\<run_id>\metrics.json` to the assistant → input for **Phase 4**
   (evaluation report + M1 gate measurement).
5. You can stop/delete the Colab runtime; the zip + Drive copy hold everything.